# Parsing statistics

<!-- Drafted with the help of an AI assistant (Claude, Anthropic), disclosed per course rules. -->

Every number and figure on this page is computed when the book is built from `data/m2/m2_papers.csv` (one row per paper), `data/m2/m2_versions.csv` (one row per version) and `data/m2/run_summary.json`. Figures are interactive (hover, zoom, click the legend).

:::{note}
**Scope.** The numbers cover the **whole official corpus**: the 20,000 assigned papers and all their 26,491 versions, parsed in run `m2-full-01` on Google Colab (see [Performance](03-performance.ipynb) for the machine). The last section ("Does the tree keep what the source has?") is the exception: it uses a subset of the official output, and says so.
:::

In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px

DATA = Path("../data/m2")
papers = pd.read_csv(DATA / "m2_papers.csv", dtype={"arxiv_id": str})
versions = pd.read_csv(DATA / "m2_versions.csv", dtype={"arxiv_id": str})
audit = pd.read_csv(DATA / "structure_subset_audit.csv", dtype={"arxiv_id": str})
run = json.loads((DATA / "run_summary.json").read_text())
assigned = json.loads((DATA / "assigned_check.json").read_text())
print(f"{assigned['found']} papers found for {assigned['expected']} assigned IDs;",
      f"missing: {len(assigned['missing'])}, not assigned: {len(assigned['extra'])}")

## Parsing success rate

A version is **parsed** when it has a root `.tex` file and every stage ran. A version **without TeX** (PDF-only, unavailable at arXiv) is recorded as `empty`; `no_root` means `.tex` files but none with `\documentclass`; `error` means a stage failed on it; `missing` means the crawl produced no folder. The *success rate* is computed over the versions that have a TeX source; *coverage* is over all versions of the corpus, empty ones included.

In [ ]:
counts = versions["status"].value_counts()
with_tex = counts.get("parsed", 0) + counts.get("no_root", 0) + counts.get("error", 0)
summary = pd.Series(
    {
        "Papers": len(papers),
        "Versions": len(versions),
        "Parsed": int(counts.get("parsed", 0)),
        "Empty (no TeX, recorded)": int(counts.get("empty", 0)),
        "No root file": int(counts.get("no_root", 0)),
        "Errors": int(counts.get("error", 0)),
        "Missing folder": int(counts.get("missing", 0)),
        "Success rate over versions with TeX (%)": round(
            100 * counts.get("parsed", 0) / with_tex, 2
        ) if with_tex else float("nan"),
        "Coverage over all versions (%)": round(100 * counts.get("parsed", 0) / len(versions), 2),
        "Papers with no parsed version": int((papers["n_parsed"] == 0).sum()),
    },
    dtype=object,
).to_frame("Value")
summary

In [ ]:
fig = px.pie(
    counts.rename_axis("status").reset_index(name="versions"),
    names="status",
    values="versions",
    title="Outcome of every version",
    hole=0.4,
)
fig

In [ ]:
reasons = (
    versions[versions["status"] != "parsed"]
    .groupby(["status", "reason"], dropna=False)
    .size()
    .rename("versions")
    .reset_index()
    .sort_values("versions", ascending=False)
)
reasons

The nine `no_root` versions have `.tex` files but none declares `\documentclass`; they are listed here so that they can be inspected.

In [ ]:
versions.loc[versions["status"] == "no_root", ["arxiv_id", "version", "n_files", "reason"]]

## Version coverage per publication

In [ ]:
papers["share_parsed"] = papers["n_parsed"] / papers["n_versions"]
fig = px.histogram(
    papers,
    x="n_versions",
    color=papers["share_parsed"].map(
        lambda s: "all versions parsed" if s == 1 else ("none parsed" if s == 0 else "some parsed")
    ),
    labels={"n_versions": "Versions of the publication", "color": "Coverage"},
    title="Versions per publication, by how many were parsed",
    log_y=True,
)
fig.update_layout(bargap=0.1, yaxis_title="Publications (log scale)")
fig

In [ ]:
pd.DataFrame(
    {
        "Publications": papers.groupby("n_versions").size(),
        "All versions parsed": papers[papers["share_parsed"] == 1].groupby("n_versions").size(),
        "No version parsed": papers[papers["share_parsed"] == 0].groupby("n_versions").size(),
    }
).fillna(0).astype(int)

## Elements and references before and after deduplication

*Before* is what the trees of all versions hold (every sentence, formula, figure and heading of every version, the document roots excluded; every reference entry of every version). *After* is what the tables store: equal elements of different versions share one row, and references that are the same work are united.

In [ ]:
totals = pd.DataFrame(
    {
        "Before deduplication": [papers["elements_before"].sum(), papers["bibtex_before"].sum()],
        "After deduplication": [
            papers["elements"].sum() - versions["status"].ne("missing").sum(),
            papers["bibtex"].sum(),
        ],
    },
    index=["Elements (roots excluded)", "Bibliography entries"],
)
totals["Reduction (%)"] = (100 * (1 - totals.iloc[:, 1] / totals.iloc[:, 0])).round(1)
totals

In [ ]:
multi = papers[papers["n_parsed"] > 1].copy()
multi["roots"] = multi["n_versions"] - multi["n_missing"]
multi["kept"] = (multi["elements"] - multi["roots"]) / multi["elements_before"]
fig = px.histogram(
    multi,
    x="kept",
    nbins=40,
    labels={"kept": "Elements stored / elements found (1 = nothing shared)"},
    title=f"Papers with several parsed versions (n = {len(multi):,}): share of elements stored",
)
fig.update_layout(yaxis_title="Publications")
fig

## References

A reference list comes from `\bibitem`, from a `.bib` file the paper names, or from a `.bbl` (the `.bbl` files that the recovery step brought back, see the [methodology](01-methodology.md)). The table counts the parsed versions by what they ended with.

In [ ]:
parsed = versions[versions["status"] == "parsed"]
no_refs = parsed["n_refs"].fillna(0).eq(0)
read_bbl = parsed["n_bbl"].fillna(0).gt(0)
named_bib_missing = parsed["missing_bib"].notna()
refs_summary = pd.Series(
    {
        "Parsed versions": len(parsed),
        "... with at least one reference": int((~no_refs).sum()),
        "... without any reference": int(no_refs.sum()),
        "Share of parsed versions without any reference (%)": round(100 * no_refs.mean(), 2),
        "Versions that read at least one .bbl": int(read_bbl.sum()),
        "... of which still without a reference": int((read_bbl & no_refs).sum()),
        "Versions that name a .bib the corpus does not have": int(named_bib_missing.sum()),
        "... of which without a reference": int((named_bib_missing & no_refs).sum()),
        "Reference entries before deduplication": int(parsed["n_refs"].fillna(0).sum()),
        "Reference entries after deduplication": int(papers["bibtex"].sum()),
    },
    dtype=object,
).to_frame("Value")
refs_summary

## Parsed versions with very few elements

A parsed version with almost no element is a sign that the root file is a stub or that a structure was not understood. They are listed so that they can be inspected; they are counted as parsed above.

In [ ]:
few = parsed[parsed["n_nodes"].fillna(0) < 20]
print(f"{len(few)} of {len(parsed):,} parsed versions have fewer than 20 elements; "
      f"{int((parsed['n_nodes'].fillna(0) == 0).sum())} have none")
parsed["n_nodes"].describe(percentiles=[0.01, 0.5, 0.99]).round(0).to_frame("elements per parsed version")

In [ ]:
few[["arxiv_id", "version", "n_files", "n_nodes", "n_refs", "root"]].sort_values("n_nodes").head(15)

## Does the tree keep what the source has?

The official output was compared with source for **2,407 parsed versions from 1,079 papers** (1,114 papers have source available) from `multi-thien-0000-4999` (`qa-thien-01`); 22,625 parsed versions outside the selected source run were skipped. The comparison counts headings, displayed formulas, floats and list items using regular expressions and reports exact count agreement per version. Differences warrant investigation; agreement does not establish semantic correctness. This selected multi-version run is not representative of the full corpus.


In [ ]:
rates = pd.DataFrame({"Versions where counts agree": [int(audit[f"{m}_agree"].sum()) for m in ["headings", "formulas", "floats", "items"]], "Versions compared": len(audit)}, index=["Headings", "Displayed formulas", "Floats", "List items"])
rates["Agreement (%)"] = (100 * rates.iloc[:, 0] / rates.iloc[:, 1]).round(1)
rates


In [ ]:
fig = px.bar(rates.reset_index(names="Structure"), x="Structure", y="Agreement (%)", text="Agreement (%)", title="Count agreement in source subset (2,407 versions)")
fig.update_yaxes(range=[0, 100])
fig.show()
